# 프로젝트 루트 찾기 — 수도코드에서 모듈까지

현재 작업 위치에서 상위 폴더를 탐색한 뒤, 같은 기능을 함수와 공용 모듈로 재사용한다.

## STEP 1. 수도코드

```text
현재 폴더에서 시작한다.
data 폴더와 pyproject.toml이 함께 있으면 프로젝트 루트로 결정한다.
없으면 부모 폴더로 올라가 반복한다.
최상위까지 찾지 못하면 오류를 발생시킨다.
```

In [1]:
from pathlib import Path
import sys

current = Path.cwd().resolve()
start = current
visited = []

while True:
    visited.append(current)
    if (current / 'data').is_dir() and (current / 'pyproject.toml').is_file():
        inline_root = current
        break
    if current.parent == current:
        raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')
    current = current.parent

print('Python:', sys.executable)
print('시작 위치:', start)
print('탐색 단계:', len(visited))
print('프로젝트 루트:', inline_root)

Python: /home/gem/research/프로그램 모음/venvs/llm-data-analysis/bin/python
시작 위치: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/path_utils_practice/notebooks/ch04
탐색 단계: 4
프로젝트 루트: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study


## STEP 2. 함수로 재사용

`return`으로 탐색을 끝내면서 찾은 경로를 호출한 곳에 돌려준다.

In [2]:
def find_project_root(start_path=None):
    current = Path(start_path or Path.cwd()).resolve()

    while True:
        if (current / 'data').is_dir() and (current / 'pyproject.toml').is_file():
            return current
        if current.parent == current:
            raise FileNotFoundError('프로젝트 루트를 찾을 수 없습니다.')
        current = current.parent

function_root = find_project_root()
print('함수 결과:', function_root)

함수 결과: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study


## STEP 3. 설치된 공용 모듈 사용

`course_utils.paths`는 `Path.cwd()`가 아니라 모듈 자신의 `__file__`에서 출발한다.

In [3]:
from course_utils.paths import get_data_dir, get_project_root

module_root = get_project_root()
data_dir = get_data_dir()

print('모듈 위치:', Path(sys.modules['course_utils.paths'].__file__).resolve())
print('프로젝트 루트:', module_root)
print('데이터 폴더:', data_dir)

모듈 위치: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/course_utils/paths.py
프로젝트 루트: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study
데이터 폴더: /home/gem/research/학교 수업/LLM 수업/llm-data-analysis-study/data


## STEP 4. 최종 검증

In [4]:
assert inline_root == function_root == module_root
assert data_dir == module_root / 'data'
assert data_dir.is_dir()
assert (module_root / 'pyproject.toml').is_file()
print('모든 경로 검증: PASS')

모든 경로 검증: PASS


## 정리

- **관찰:** 깊은 Notebook 폴더에서 실행해도 세 방식이 동일한 프로젝트 루트를 반환했다.
- **판단:** 반복 사용하는 코드에는 현재 작업 위치보다 모듈의 `__file__`을 기준으로 한 공용 함수를 사용한다.
- **의미:** Chapter별 상대경로 계산과 `sys.path.insert()` 중복을 제거할 수 있다.
- **한계:** 프로젝트 루트를 식별하는 `data`와 `pyproject.toml` 구조가 바뀌면 판정 조건도 수정해야 한다.